# Simulating P vs. G

:::{sidebar} Discovering Game Theory
This course uses an **online textbook**: [Game Theory, a Discovery Approach](https://nordstrommath.com/IntroGameTheory2e/intro_game_theory.html), by Jennifer Nordstrom.
:::

Following on from the previous section, we would like to be able to construct a way to simulate the winning percentages of the various strategies. The simulation functions are described below.

In [2]:
## Do not change this cell, only execute it. 
## This cell initializes Python so that pandas, numpy and scipy packages are ready to use.

from datascience import *
import random
from IPython.display import display, Markdown
import pandas
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import scipy as sp
import math
import scipy.stats as stats

## Intial Values
We first initialize the values we need:
- $p$ : number of police units
- $g$ : number of guerrilla units
- strat : an arrays of strategies listed by larger number of the two groups of units
- index : the column or row numbers in the results table

In [3]:
p = 7
g = 5

### Dictionaries

Below, we create a function that creates player dictionaries to keep track of row-column indexes for the results table and strategies listed by integer value.

In [21]:
def create_player_dict(name, anchor_val):
    strat = np.arange(math.ceil(anchor_val / 2), anchor_val + 1)
    index = np.arange(1, len(strat) + 1)
    strat_name = [f"{x} - {anchor_val - x}" for x in strat]
    return {
        'player': name,
        'index': index,
        'strat': strat,
        'strat_name': strat_name
    }

In [22]:
anchor_val = 5
strat = np.arange(math.ceil(anchor_val / 2), anchor_val + 1)
print(strat)
strat_name = [f"{x} - {anchor_val - x}" for x in strat]
print(strat_name)

[3 4 5]
['3 - 2', '4 - 1', '5 - 0']


In [23]:
# Creating the dictionaries
P = create_player_dict('P', p)
G = create_player_dict('G', g)
G['index'] = G['index'] - 1

In [24]:
# Displaying the results
print(P)
print(G)

{'player': 'P', 'index': array([1, 2, 3, 4]), 'strat': array([4, 5, 6, 7]), 'strat_name': ['4 - 3', '5 - 2', '6 - 1', '7 - 0']}
{'player': 'G', 'index': array([0, 1, 2]), 'strat': array([3, 4, 5]), 'strat_name': ['3 - 2', '4 - 1', '5 - 0']}


We can also display the indexes for a single player or the strategies for that player.

In [25]:
print(P['strat']) ; print(P['index']) ; print(G['strat']) ; print(G['index'])

[4 5 6 7]
[1 2 3 4]
[3 4 5]
[0 1 2]


### Init() Function
To initialize the results table, we wish to play each column strategy against each row strategy. We plan for 40 simulated battles with $10$ ghost results.  With $5$ ghost wins and $5$ ghost losses, the win percentages will skew to the center and away from the extremes of $0$ and $1$. The simulation results appear as though $50$ real results were simulated.

Adding ghost results is a traditional way to initialize proportions when we know some of those proporitions may be close to $0$ or $1$.

In [26]:
def init(p_max, g_max, p_strat, g_strat):
   A = p_strat ; B = p_max - A ;
   res = ()
   for k in range(40):
      a = np.random.choice(make_array( g_strat, g_max - g_strat ) ) ; b = g_strat - a ;
      if ( ( a > A ) or ( b > B ) ):
         v = 0
      else:
            v = 1
      res = np.append(res, v)
   temp = ( sum(res) + 5 ) / 50
   t = round(temp,3)
   return t

In [29]:
print(G['strat'])
G['strat_name']

[3 4 5]


['3 - 2', '4 - 1', '5 - 0']

### Results Table

The following code simply sets up a table in which we plan to store the win percentages for each strategy matchup. The values in the table are irrelevant to the actual game and will serve as placeholders.

In [59]:
results = Table().with_columns('', G['strat_name'])
for t in P['index']:
    results = results.with_columns(P['strat_name'][t-1],np.arange(len(G['strat'])).astype(np.float64)+t)
for t in P['index']:
  for s in G['index']:
    results[t][s] = init(p,g,P['strat'][t-1],G['strat'][s])

#display(Markdown(f"<b>{p} Police vs. {g} Guerillas</b>"))
results

,4 - 3,5 - 2,6 - 1,7 - 0
3 - 2,0.9,0.9,0.9,0.46
4 - 1,0.9,0.44,0.5,0.66
5 - 0,0.1,0.42,0.48,0.5


The code to this point has created a Police vs. Guerrillas game for 7P vs. 5G. Now, we need a function to select a random strategy for each player, play those strategies against each other, and compute the exponential moving average (ema) of the win percentage.

In [60]:
def play(p_0, g_0):
    A = np.random.choice(P['strat']) ; B = p_0 - A
    a = np.random.choice(G['strat']) ; b = g_0 - a
    if (a > A) or (b > B):
        v = 0
    else:
        v = 1
    alpha = 0.2
    current_ema = results[p_0][g_0]
    value = (v * alpha) + (current_ema * (1 - alpha))
    results[p_0][g_0] = value
    print(A)
    print(a)
    return value  # This returns a "tuple" of three values

In [62]:
val = play(3,2)
val

4
4


0.58400000000000007

In [63]:
results

,4 - 3,5 - 2,6 - 1,7 - 0
3 - 2,0.9,0.9,0.9,0.46
4 - 1,0.9,0.44,0.5,0.66
5 - 0,0.1,0.42,0.584,0.5


When done, this function should be able to iterate the play function thousands of times. If the next strategy choices are randomly chosen based upon past winning percentage probabilities, the simulation will iterate to a table showing that converges to the correct solution for the game matrix shown.